# Нейросетевое моделирование с учётом отдельных участков

Этот ноутбук проверяет, насколько постановка задачи зависит от различий между участками.

В предыдущем ноутбуке модели обучались на объединённой таблице интервальных наблюдений. Такая постановка полезна как общий базовый эксперимент, но она смешивает разные участки, у которых может быть разная динамика изменения береговой бровки.

В этом ноутбуке выполняются три проверки:

1. сравнение нейросетевой модели без учёта участка и с учётом `site_id`;
2. обучение отдельных нейросетевых моделей для участков, где достаточно наблюдений;
3. построение графика фактических и предсказанных значений для выбранного участка.

Цель ноутбука — показать, что для неоднородных экологических данных важно учитывать пространственную неоднородность участков.


In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.analysis.baseline_modeling import choose_target, build_feature_frame, build_models
from src.parsers.common import PROCESSED_DIR, REPORTS_DIR

warnings.filterwarnings("ignore", category=UserWarning)

TABLES_DIR = REPORTS_DIR / "tables"
FIGURES_DIR = REPORTS_DIR / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

dataset_path = PROCESSED_DIR / "final_dataset_for_modeling.csv"
data = pd.read_csv(dataset_path)

target = choose_target(data)

print("Датасет:", dataset_path)
print("Строк:", len(data))
print("Целевая переменная:", target)


## 1. Распределение наблюдений по участкам

Сначала проверяется, сколько наблюдений есть по каждому участку. Это важно, потому что отдельную модель нельзя корректно обучать на слишком малом числе строк.

Если по участку мало наблюдений, его лучше использовать только в общей модели, а не строить для него отдельную нейросетевую модель.


In [ ]:
site_summary = (
    data
    .groupby("site_id")
    .agg(
        n_rows=("site_id", "size"),
        target_mean=(target, "mean"),
        target_median=(target, "median"),
        target_max=(target, "max"),
    )
    .reset_index()
    .sort_values("n_rows", ascending=False)
)

site_summary_path = TABLES_DIR / "06_site_observation_summary.csv"
site_summary.to_csv(site_summary_path, index=False)

site_summary


In [ ]:
top_sites = site_summary.head(15).sort_values("n_rows")

fig, ax = plt.subplots(figsize=(10, 6))

ax.barh(top_sites["site_id"], top_sites["n_rows"])

ax.set_title("Количество интервальных наблюдений по участкам")
ax.set_xlabel("Число наблюдений")
ax.set_ylabel("Участок")
ax.grid(axis="x", alpha=0.3)

output_path = FIGURES_DIR / "06_site_observation_counts.png"
fig.savefig(output_path, dpi=240, bbox_inches="tight", facecolor="white")

plt.show()

print("Сохранено:", output_path)


## 2. Общая нейросетевая модель без учёта и с учётом участка

В этом блоке сравниваются четыре варианта нейросетевой модели:

1. обычная `MLPRegressor` без `site_id`;
2. `MLPRegressor` без `site_id`, но с логарифмическим преобразованием целевой переменной;
3. `MLPRegressor` с добавлением `site_id` как категориального признака;
4. `MLPRegressor` с `site_id` и логарифмическим преобразованием целевой переменной.

Это позволяет проверить, помогает ли модели информация о принадлежности наблюдения к конкретному участку.


In [ ]:
def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    return {
        "test_mae": mean_absolute_error(y_true, y_pred),
        "test_rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "test_r2": r2_score(y_true, y_pred),
    }


def fit_predict_mlp(X, y, numeric_features, categorical_features, use_log_target, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=random_state,
    )

    models = build_models(numeric_features, categorical_features)
    model = clone(models["MLPRegressor"])

    if use_log_target:
        model = TransformedTargetRegressor(
            regressor=model,
            func=np.log1p,
            inverse_func=np.expm1,
        )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred = np.clip(y_pred, 0, None)

    metrics = regression_metrics(y_test, y_pred)

    return metrics, y_test, y_pred


In [ ]:
X_base, y_base, metadata_base, numeric_features_base, categorical_features_base, excluded_columns_base = build_feature_frame(data, target)

global_results = []

metrics, y_test_base, y_pred_base = fit_predict_mlp(
    X_base,
    y_base,
    numeric_features_base,
    categorical_features_base,
    use_log_target=False,
)

global_results.append(
    {
        "variant": "MLP без site_id",
        "n_rows": len(X_base),
        **metrics,
    }
)

metrics, y_test_log, y_pred_log = fit_predict_mlp(
    X_base,
    y_base,
    numeric_features_base,
    categorical_features_base,
    use_log_target=True,
)

global_results.append(
    {
        "variant": "MLP без site_id + log1p(y)",
        "n_rows": len(X_base),
        **metrics,
    }
)

X_site = X_base.copy()
X_site["site_id_feature"] = metadata_base["site_id"].astype(str).to_numpy()

numeric_features_site = list(numeric_features_base)
categorical_features_site = list(categorical_features_base) + ["site_id_feature"]

metrics, y_test_site, y_pred_site = fit_predict_mlp(
    X_site,
    y_base,
    numeric_features_site,
    categorical_features_site,
    use_log_target=False,
)

global_results.append(
    {
        "variant": "MLP с site_id",
        "n_rows": len(X_site),
        **metrics,
    }
)

metrics, y_test_site_log, y_pred_site_log = fit_predict_mlp(
    X_site,
    y_base,
    numeric_features_site,
    categorical_features_site,
    use_log_target=True,
)

global_results.append(
    {
        "variant": "MLP с site_id + log1p(y)",
        "n_rows": len(X_site),
        **metrics,
    }
)

global_metrics = pd.DataFrame(global_results).sort_values("test_mae")

global_metrics_path = TABLES_DIR / "06_global_siteaware_mlp_metrics.csv"
global_metrics.to_csv(global_metrics_path, index=False)

global_metrics


In [ ]:
plot_df = global_metrics.sort_values("test_mae", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))

ax.barh(plot_df["variant"], plot_df["test_mae"])

ax.set_title("Сравнение вариантов нейросетевой модели")
ax.set_xlabel("MAE, м/год")
ax.set_ylabel("Вариант модели")
ax.grid(axis="x", alpha=0.3)

for index, value in enumerate(plot_df["test_mae"]):
    ax.text(value, index, f" {value:.3f}", va="center")

output_path = FIGURES_DIR / "06_global_siteaware_mlp_mae.png"
fig.savefig(output_path, dpi=240, bbox_inches="tight", facecolor="white")

plt.show()

print("Сохранено:", output_path)


## 3. Отдельные нейросетевые модели по участкам

В этом блоке модель обучается отдельно для каждого участка, если по нему достаточно наблюдений.

Такая постановка не пытается построить универсальную модель для всех берегов сразу. Она проверяет, может ли нейросеть лучше описывать данные внутри отдельного участка, где условия более однородны.


In [ ]:
MIN_SITE_ROWS = 25
RANDOM_STATE = 42

eligible_sites = (
    site_summary
    .loc[site_summary["n_rows"] >= MIN_SITE_ROWS, "site_id"]
    .tolist()
)

eligible_sites


In [ ]:
def evaluate_single_site(site_id, min_rows=MIN_SITE_ROWS, random_state=RANDOM_STATE):
    site_data = data.loc[data["site_id"].eq(site_id)].copy()

    if len(site_data) < min_rows:
        return None, None

    try:
        site_target = choose_target(site_data)
        X_site_local, y_site_local, metadata_site_local, numeric_site_local, categorical_site_local, excluded_site_local = build_feature_frame(
            site_data,
            site_target,
        )
    except Exception:
        return None, None

    if len(X_site_local) < min_rows or pd.Series(y_site_local).nunique() < 3:
        return None, None

    X_train, X_test, y_train, y_test, metadata_train, metadata_test = train_test_split(
        X_site_local,
        y_site_local,
        metadata_site_local,
        test_size=0.25,
        random_state=random_state,
    )

    if len(X_train) < 10 or len(X_test) < 5:
        return None, None

    models = build_models(numeric_site_local, categorical_site_local)

    mlp = TransformedTargetRegressor(
        regressor=clone(models["MLPRegressor"]),
        func=np.log1p,
        inverse_func=np.expm1,
    )

    dummy = DummyRegressor(strategy="median")

    mlp.fit(X_train, y_train)
    dummy.fit(X_train, y_train)

    mlp_pred = np.clip(mlp.predict(X_test), 0, None)
    dummy_pred = np.clip(dummy.predict(X_test), 0, None)

    mlp_metrics = regression_metrics(y_test, mlp_pred)
    dummy_metrics = regression_metrics(y_test, dummy_pred)

    result = {
        "site_id": site_id,
        "n_rows": len(X_site_local),
        "n_train": len(X_train),
        "n_test": len(X_test),
        "target_mean": float(pd.Series(y_site_local).mean()),
        "target_median": float(pd.Series(y_site_local).median()),
        "dummy_mae": dummy_metrics["test_mae"],
        "mlp_log_mae": mlp_metrics["test_mae"],
        "dummy_rmse": dummy_metrics["test_rmse"],
        "mlp_log_rmse": mlp_metrics["test_rmse"],
        "dummy_r2": dummy_metrics["test_r2"],
        "mlp_log_r2": mlp_metrics["test_r2"],
        "mae_improvement_vs_dummy": dummy_metrics["test_mae"] - mlp_metrics["test_mae"],
    }

    predictions = metadata_test.reset_index(drop=True).copy()
    predictions["site_id"] = site_id
    predictions["actual"] = np.asarray(y_test)
    predictions["predicted_mlp_log"] = np.asarray(mlp_pred)
    predictions["predicted_dummy"] = np.asarray(dummy_pred)
    predictions["absolute_error_mlp_log"] = np.abs(predictions["actual"] - predictions["predicted_mlp_log"])

    return result, predictions


site_results = []
site_prediction_tables = []

for site_id in eligible_sites:
    result, predictions = evaluate_single_site(site_id)

    if result is not None:
        site_results.append(result)
        site_prediction_tables.append(predictions)

site_metrics = pd.DataFrame(site_results)

if site_prediction_tables:
    site_predictions = pd.concat(site_prediction_tables, ignore_index=True)
else:
    site_predictions = pd.DataFrame()

site_metrics_path = TABLES_DIR / "06_site_specific_mlp_metrics.csv"
site_predictions_path = TABLES_DIR / "06_site_specific_mlp_predictions.csv"

site_metrics.to_csv(site_metrics_path, index=False)
site_predictions.to_csv(site_predictions_path, index=False)

site_metrics.sort_values("mlp_log_mae")


In [ ]:
if len(site_metrics) > 0:
    plot_df = (
        site_metrics
        .sort_values("n_rows", ascending=False)
        .head(12)
        .sort_values("mlp_log_mae", ascending=True)
    )

    x = np.arange(len(plot_df))
    width = 0.35

    fig, ax = plt.subplots(figsize=(12, 5))

    ax.bar(
        x - width / 2,
        plot_df["dummy_mae"],
        width=width,
        label="DummyRegressor",
    )

    ax.bar(
        x + width / 2,
        plot_df["mlp_log_mae"],
        width=width,
        label="MLPRegressor + log1p(y)",
    )

    ax.set_title("Отдельные модели по участкам: сравнение MAE")
    ax.set_xlabel("Участок")
    ax.set_ylabel("MAE, м/год")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["site_id"], rotation=35, ha="right")
    ax.grid(axis="y", alpha=0.3)
    ax.legend()

    output_path = FIGURES_DIR / "06_site_specific_mlp_mae.png"
    fig.savefig(output_path, dpi=240, bbox_inches="tight", facecolor="white")

    plt.show()

    print("Сохранено:", output_path)
else:
    print("Недостаточно участков для построения отдельных моделей.")


## 4. Фактические и предсказанные значения для выбранного участка

Для наглядности выбирается участок, на котором отдельная нейросетевая модель дала наибольшее улучшение MAE относительно медианной базовой модели.

На графике фактические и предсказанные значения показаны для тестовых наблюдений выбранного участка.


In [ ]:
if len(site_metrics) == 0 or len(site_predictions) == 0:
    selected_site = None
    print("Нет данных для выбора участка.")
else:
    selected_site = (
        site_metrics
        .sort_values(["mae_improvement_vs_dummy", "n_test"], ascending=[False, False])
        .iloc[0]["site_id"]
    )

    selected_predictions = (
        site_predictions
        .loc[site_predictions["site_id"].eq(selected_site)]
        .copy()
        .sort_values("actual")
        .reset_index(drop=True)
    )

    selected_row = site_metrics.loc[site_metrics["site_id"].eq(selected_site)].iloc[0]

    print("Выбранный участок:", selected_site)
    print(f"MAE Dummy: {selected_row['dummy_mae']:.3f}")
    print(f"MAE MLP: {selected_row['mlp_log_mae']:.3f}")
    print(f"Улучшение MAE: {selected_row['mae_improvement_vs_dummy']:.3f}")

    selected_predictions


In [ ]:
if selected_site is not None:
    fig, ax = plt.subplots(figsize=(11, 5))

    ax.plot(
        selected_predictions.index,
        selected_predictions["actual"],
        marker="o",
        linewidth=2,
        label="Фактическое значение",
    )

    ax.plot(
        selected_predictions.index,
        selected_predictions["predicted_mlp_log"],
        marker="o",
        linewidth=2,
        label="Прогноз нейросетевой модели",
    )

    ax.set_title(f"Участок {selected_site}: факт и прогноз нейросетевой модели")
    ax.set_xlabel("Тестовые наблюдения участка, отсортированные по фактическому значению")
    ax.set_ylabel("Интенсивность изменения бровки, м/год")
    ax.grid(alpha=0.3)
    ax.legend()

    metrics_text = (
        f"MAE = {selected_row['mlp_log_mae']:.3f}\n"
        f"RMSE = {selected_row['mlp_log_rmse']:.3f}\n"
        f"R² = {selected_row['mlp_log_r2']:.3f}"
    )

    ax.text(
        0.02,
        0.95,
        metrics_text,
        transform=ax.transAxes,
        verticalalignment="top",
        bbox=dict(boxstyle="round", alpha=0.15),
    )

    output_path = FIGURES_DIR / "06_selected_site_mlp_actual_vs_predicted.png"
    fig.savefig(output_path, dpi=240, bbox_inches="tight", facecolor="white")

    plt.show()

    print("Сохранено:", output_path)


## 5. Интерпретация

Этот ноутбук проверяет, насколько нейросетевая модель чувствительна к различиям между участками.

Если модель с `site_id` показывает меньшее значение MAE, чем модель без `site_id`, это означает, что принадлежность к участку содержит важную информацию для прогноза. В таком случае объединённая модель без учёта участка слишком сильно смешивает разные береговые условия.

Если отдельные модели по участкам дают лучшее качество, чем простая медианная базовая модель, это показывает, что внутри отдельных участков нейросеть может извлекать локальный предсказательный сигнал.

Если улучшение не наблюдается, это тоже важный результат: доступных признаков и числа наблюдений недостаточно для устойчивого локального нейросетевого прогноза.

Главный смысл эксперимента состоит не в замене предыдущего моделирования, а в проверке пространственной неоднородности данных. Для отчёта этот блок можно использовать как дополнительное обоснование того, что экологические данные нельзя рассматривать как полностью однородную таблицу.


## 6. Подбор участка для наглядного временного прогноза

Обычный график по всем строкам участка может выглядеть плохо, если в один год попадает несколько профилей с разными значениями. Тогда линия делает вертикальные скачки, хотя это не ошибка модели, а особенность структуры данных.

Для более наглядного графика наблюдения агрегируются по дате окончания интервала. Для каждой даты берётся медианное фактическое значение и медианный прогноз модели.

Такой график не подменяет исходные данные, а показывает обобщённую динамику участка без визуального шума от нескольких профилей в один момент времени.


In [ ]:
def build_temporal_forecast_for_site(site_id, test_fraction=0.25):
    site_data_time = data.loc[data["site_id"].eq(site_id)].copy()

    site_data_time["date_end"] = pd.to_datetime(site_data_time["date_end"], errors="coerce")
    site_data_time["date_start"] = pd.to_datetime(site_data_time["date_start"], errors="coerce")

    site_data_time = site_data_time.dropna(subset=["date_end"]).sort_values("date_end").reset_index(drop=True)

    site_target = choose_target(site_data_time)

    X_time, y_time, metadata_time, numeric_time, categorical_time, excluded_time = build_feature_frame(
        site_data_time,
        site_target,
    )

    n_total = len(X_time)
    n_test = max(5, int(round(n_total * test_fraction)))
    n_train = n_total - n_test

    if n_train < 10 or n_test < 5:
        return None

    X_train_time = X_time.iloc[:n_train].copy()
    X_test_time = X_time.iloc[n_train:].copy()

    if hasattr(y_time, "iloc"):
        y_train_time = y_time.iloc[:n_train].copy()
        y_test_time = y_time.iloc[n_train:].copy()
    else:
        y_train_time = y_time[:n_train]
        y_test_time = y_time[n_train:]

    models_time = build_models(numeric_time, categorical_time)

    mlp_time = TransformedTargetRegressor(
        regressor=clone(models_time["MLPRegressor"]),
        func=np.log1p,
        inverse_func=np.expm1,
    )

    mlp_time.fit(X_train_time, y_train_time)
    y_pred_time = np.clip(mlp_time.predict(X_test_time), 0, None)

    mae_time = mean_absolute_error(y_test_time, y_pred_time)
    rmse_time = np.sqrt(mean_squared_error(y_test_time, y_pred_time))
    r2_time = r2_score(y_test_time, y_pred_time) if len(y_test_time) >= 2 else np.nan

    time_plot_df = metadata_time.reset_index(drop=True).copy()
    time_plot_df["actual"] = np.asarray(y_time)
    time_plot_df["predicted"] = np.nan
    time_plot_df.loc[n_train:, "predicted"] = y_pred_time
    time_plot_df["date_end"] = pd.to_datetime(time_plot_df["date_end"], errors="coerce")

    aggregated = (
        time_plot_df
        .groupby("date_end", as_index=False)
        .agg(
            actual=("actual", "median"),
            predicted=("predicted", "median"),
            n_rows=("actual", "size"),
        )
        .sort_values("date_end")
        .reset_index(drop=True)
    )

    pred_part = aggregated.loc[aggregated["predicted"].notna()].copy()

    if len(pred_part) < 3:
        return None

    return {
        "site_id": site_id,
        "n_total": n_total,
        "n_train": n_train,
        "n_test": n_test,
        "n_dates": len(aggregated),
        "n_forecast_dates": len(pred_part),
        "mae": mae_time,
        "rmse": rmse_time,
        "r2": r2_time,
        "data": time_plot_df,
        "aggregated": aggregated,
    }


In [ ]:
candidate_sites = (
    site_metrics
    .loc[site_metrics["n_test"] >= 5]
    .copy()
)

candidate_sites = candidate_sites.loc[candidate_sites["mlp_log_mae"].notna()].copy()

candidate_sites["r2_for_sort"] = candidate_sites["mlp_log_r2"].fillna(-999)

candidate_sites = candidate_sites.sort_values(
    ["mlp_log_mae", "r2_for_sort", "n_rows"],
    ascending=[True, False, False],
)

candidate_site_ids = candidate_sites["site_id"].head(12).tolist()

temporal_candidates = []

for site_id in candidate_site_ids:
    result = build_temporal_forecast_for_site(site_id)
    if result is not None:
        temporal_candidates.append(result)

candidate_summary = pd.DataFrame(
    [
        {
            "site_id": item["site_id"],
            "n_total": item["n_total"],
            "n_train": item["n_train"],
            "n_test": item["n_test"],
            "n_dates": item["n_dates"],
            "n_forecast_dates": item["n_forecast_dates"],
            "mae": item["mae"],
            "rmse": item["rmse"],
            "r2": item["r2"],
        }
        for item in temporal_candidates
    ]
)

candidate_summary_path = TABLES_DIR / "06_temporal_forecast_candidate_sites.csv"
candidate_summary.to_csv(candidate_summary_path, index=False)

candidate_summary.sort_values(["mae", "r2"], ascending=[True, False])


## 7. Галерея кандидатов

Ниже строятся несколько временных графиков для участков-кандидатов. Это нужно для выбора участка, который одновременно имеет приемлемые метрики и читаемый график.

Для отчёта лучше брать не обязательно участок с минимальным MAE, а участок, где график визуально понятен и тестовое окно содержит несколько дат.


In [ ]:
max_plots = min(6, len(temporal_candidates))

if max_plots == 0:
    raise ValueError("Нет подходящих участков для временного графика.")

fig, axes = plt.subplots(max_plots, 1, figsize=(12, 4 * max_plots), sharex=False)

if max_plots == 1:
    axes = [axes]

for ax, item in zip(axes, temporal_candidates[:max_plots]):
    aggregated = item["aggregated"]
    pred_part = aggregated.loc[aggregated["predicted"].notna()].copy()

    ax.plot(
        aggregated["date_end"],
        aggregated["actual"],
        marker="o",
        linewidth=2,
        label="Факт",
    )

    ax.plot(
        pred_part["date_end"],
        pred_part["predicted"],
        marker="s",
        linewidth=2,
        linestyle="--",
        label="Прогноз",
    )

    ax.axvspan(
        pred_part["date_end"].min(),
        pred_part["date_end"].max(),
        alpha=0.12,
        label="Окно прогноза",
    )

    ax.axvline(
        pred_part["date_end"].min(),
        linestyle=":",
        linewidth=2,
        alpha=0.8,
    )

    ax.set_title(
        f"{item['site_id']} | MAE={item['mae']:.3f}, RMSE={item['rmse']:.3f}, R²={item['r2']:.3f}"
    )

    ax.set_ylabel("м/год")
    ax.grid(alpha=0.3)
    ax.legend(loc="best")

axes[-1].set_xlabel("Дата окончания интервала")

output_path = FIGURES_DIR / "06_temporal_forecast_candidate_gallery.png"
fig.savefig(output_path, dpi=240, bbox_inches="tight", facecolor="white")

plt.show()

print("Сохранено:", output_path)


## 8. Финальный временной график для отчёта

В следующей ячейке можно вручную указать участок для финального графика.

Если оставить `MANUAL_SITE_ID = None`, будет выбран первый участок из списка кандидатов. Если какой-то участок из галереи выглядит лучше, нужно вписать его идентификатор строкой, например:

`MANUAL_SITE_ID = "berezhnovka"`


In [ ]:
MANUAL_SITE_ID = None

if MANUAL_SITE_ID is None:
    selected_item = temporal_candidates[0]
else:
    matches = [item for item in temporal_candidates if item["site_id"] == MANUAL_SITE_ID]
    if not matches:
        selected_item = build_temporal_forecast_for_site(MANUAL_SITE_ID)
        if selected_item is None:
            raise ValueError(f"Не удалось построить временной прогноз для участка: {MANUAL_SITE_ID}")
    else:
        selected_item = matches[0]

selected_site_for_time_plot = selected_item["site_id"]
aggregated_time_plot = selected_item["aggregated"]
pred_part = aggregated_time_plot.loc[aggregated_time_plot["predicted"].notna()].copy()

mae_time = selected_item["mae"]
rmse_time = selected_item["rmse"]
r2_time = selected_item["r2"]

time_predictions_path = TABLES_DIR / "06_selected_site_temporal_forecast.csv"
aggregated_time_plot.to_csv(time_predictions_path, index=False)

print("Выбранный участок:", selected_site_for_time_plot)
print("Всего строк:", selected_item["n_total"])
print("Тестовых строк:", selected_item["n_test"])
print("Дат на графике:", selected_item["n_dates"])
print("Дат в окне прогноза:", selected_item["n_forecast_dates"])
print(f"MAE = {mae_time:.3f}")
print(f"RMSE = {rmse_time:.3f}")
print(f"R² = {r2_time:.3f}")
print("Сохранено:", time_predictions_path)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(
    aggregated_time_plot["date_end"],
    aggregated_time_plot["actual"],
    marker="o",
    linewidth=2,
    label="Факт",
)

ax.plot(
    pred_part["date_end"],
    pred_part["predicted"],
    marker="s",
    linewidth=2,
    linestyle="--",
    label="Прогноз нейросетевой модели",
)

ax.axvspan(
    pred_part["date_end"].min(),
    pred_part["date_end"].max(),
    alpha=0.12,
    label="Окно прогноза",
)

ax.axvline(
    pred_part["date_end"].min(),
    linestyle=":",
    linewidth=2,
    alpha=0.8,
)

ax.set_title(
    f"Временной ряд и прогноз нейросетевой модели\n"
    f"Участок: {selected_site_for_time_plot}"
)
ax.set_xlabel("Дата окончания интервала")
ax.set_ylabel("Интенсивность изменения бровки, м/год")
ax.grid(alpha=0.3)
ax.legend(loc="best")

metrics_text = (
    f"MAE = {mae_time:.3f}\n"
    f"RMSE = {rmse_time:.3f}\n"
    f"R² = {r2_time:.3f}"
)

ax.text(
    0.78,
    0.95,
    metrics_text,
    transform=ax.transAxes,
    verticalalignment="top",
    bbox=dict(boxstyle="round", alpha=0.15),
)

output_path = FIGURES_DIR / "06_selected_site_temporal_forecast.png"
fig.savefig(output_path, dpi=240, bbox_inches="tight", facecolor="white")

plt.show()

print("Сохранено:", output_path)


## 9. Вывод по участковому прогнозу

Временной график показывает работу нейросетевой модели внутри одного участка, где данные более однородны, чем в объединённой таблице по всем участкам.

Для построения графика значения агрегированы по дате окончания интервала с помощью медианы. Это позволяет убрать визуальный шум от нескольких профилей в одну дату, не изменяя исходные данные.

Такой график можно использовать в отчёте как иллюстрацию локального нейросетевого прогноза. При этом результат нельзя автоматически переносить на все участки водохранилища.


## 10. Сглаженный временной график для отчёта

Интервальная скорость изменения береговой бровки является шумной величиной: в одну дату могут попадать разные профили, а отдельные интервалы дают резкие скачки.

Поэтому для отчётной иллюстрации дополнительно строится сглаженный график. Он не заменяет исходные данные и не используется для расчёта основных метрик. Его задача — показать общую тенденцию и визуально сопоставить фактическую динамику с прогнозом нейросетевой модели.

Сглаживание применяется только к агрегированным по дате значениям.


In [ ]:
def add_smoothing_columns(frame, window=3):
    result = frame.copy()

    result["actual_smooth"] = (
        result["actual"]
        .rolling(window=window, center=True, min_periods=1)
        .median()
    )

    result["predicted_smooth"] = (
        result["predicted"]
        .rolling(window=window, center=True, min_periods=1)
        .median()
    )

    return result


def smoothed_forecast_score(item, window=3):
    frame = add_smoothing_columns(item["aggregated"], window=window)
    pred_frame = frame.loc[frame["predicted"].notna()].copy()

    if len(pred_frame) < 3:
        return None

    smooth_mae = mean_absolute_error(
        pred_frame["actual_smooth"],
        pred_frame["predicted_smooth"],
    )

    smooth_rmse = np.sqrt(
        mean_squared_error(
            pred_frame["actual_smooth"],
            pred_frame["predicted_smooth"],
        )
    )

    if pred_frame["actual_smooth"].nunique() > 1 and len(pred_frame) >= 2:
        smooth_r2 = r2_score(
            pred_frame["actual_smooth"],
            pred_frame["predicted_smooth"],
        )
    else:
        smooth_r2 = np.nan

    return {
        "site_id": item["site_id"],
        "n_total": item["n_total"],
        "n_forecast_dates": item["n_forecast_dates"],
        "raw_mae": item["mae"],
        "raw_rmse": item["rmse"],
        "raw_r2": item["r2"],
        "smooth_mae": smooth_mae,
        "smooth_rmse": smooth_rmse,
        "smooth_r2": smooth_r2,
    }


smooth_candidate_rows = []

for item in temporal_candidates:
    row = smoothed_forecast_score(item, window=3)
    if row is not None:
        smooth_candidate_rows.append(row)

smooth_candidate_summary = pd.DataFrame(smooth_candidate_rows)

smooth_candidate_summary_path = TABLES_DIR / "06_smoothed_temporal_forecast_candidate_sites.csv"
smooth_candidate_summary.to_csv(smooth_candidate_summary_path, index=False)

smooth_candidate_summary.sort_values(
    ["smooth_mae", "smooth_r2", "n_forecast_dates"],
    ascending=[True, False, False],
)


## 11. Галерея сглаженных графиков

Ниже показаны несколько участков-кандидатов после сглаживания. Для отчёта лучше выбрать график, где прогноз визуально сопоставим с фактической тенденцией и при этом не основан на слишком малом числе точек.


In [ ]:
smooth_sorted_sites = (
    smooth_candidate_summary
    .sort_values(["smooth_mae", "smooth_r2", "n_forecast_dates"], ascending=[True, False, False])
    ["site_id"]
    .head(6)
    .tolist()
)

smooth_items = [
    item
    for site_id in smooth_sorted_sites
    for item in temporal_candidates
    if item["site_id"] == site_id
]

max_plots = len(smooth_items)

if max_plots == 0:
    raise ValueError("Нет подходящих участков для сглаженного графика.")

fig, axes = plt.subplots(max_plots, 1, figsize=(12, 4 * max_plots), sharex=False)

if max_plots == 1:
    axes = [axes]

for ax, item in zip(axes, smooth_items):
    frame = add_smoothing_columns(item["aggregated"], window=3)
    pred_frame = frame.loc[frame["predicted"].notna()].copy()

    score_row = smooth_candidate_summary.loc[
        smooth_candidate_summary["site_id"].eq(item["site_id"])
    ].iloc[0]

    ax.plot(
        frame["date_end"],
        frame["actual"],
        marker="o",
        linewidth=1,
        alpha=0.35,
        label="Факт, агрегированные значения",
    )

    ax.plot(
        frame["date_end"],
        frame["actual_smooth"],
        marker="o",
        linewidth=2,
        label="Факт, сглаженная тенденция",
    )

    ax.plot(
        pred_frame["date_end"],
        pred_frame["predicted_smooth"],
        marker="s",
        linewidth=2,
        linestyle="--",
        label="Прогноз, сглаженный",
    )

    ax.axvspan(
        pred_frame["date_end"].min(),
        pred_frame["date_end"].max(),
        alpha=0.12,
        label="Окно прогноза",
    )

    ax.set_title(
        f"{item['site_id']} | сглаж. MAE={score_row['smooth_mae']:.3f}, "
        f"сглаж. R²={score_row['smooth_r2']:.3f}"
    )

    ax.set_ylabel("м/год")
    ax.grid(alpha=0.3)
    ax.legend(loc="best")

axes[-1].set_xlabel("Дата окончания интервала")

output_path = FIGURES_DIR / "06_smoothed_temporal_forecast_candidate_gallery.png"
fig.savefig(output_path, dpi=240, bbox_inches="tight", facecolor="white")

plt.show()

print("Сохранено:", output_path)


## 12. Финальный сглаженный график для отчёта

В следующей ячейке можно вручную указать участок для финального сглаженного графика.

Если оставить `SMOOTH_MANUAL_SITE_ID = None`, будет выбран лучший участок по сглаженному MAE среди кандидатов. Если в галерее визуально лучше выглядит другой участок, нужно вписать его идентификатор строкой.


In [ ]:
SMOOTH_MANUAL_SITE_ID = None

if SMOOTH_MANUAL_SITE_ID is None:
    selected_smooth_site = (
        smooth_candidate_summary
        .sort_values(["smooth_mae", "smooth_r2", "n_forecast_dates"], ascending=[True, False, False])
        .iloc[0]["site_id"]
    )
else:
    selected_smooth_site = SMOOTH_MANUAL_SITE_ID

selected_smooth_matches = [
    item for item in temporal_candidates
    if item["site_id"] == selected_smooth_site
]

if not selected_smooth_matches:
    selected_smooth_item = build_temporal_forecast_for_site(selected_smooth_site)
    if selected_smooth_item is None:
        raise ValueError(f"Не удалось построить сглаженный график для участка: {selected_smooth_site}")
else:
    selected_smooth_item = selected_smooth_matches[0]

selected_smooth_frame = add_smoothing_columns(selected_smooth_item["aggregated"], window=3)
selected_smooth_pred = selected_smooth_frame.loc[selected_smooth_frame["predicted"].notna()].copy()

selected_smooth_metrics = smoothed_forecast_score(selected_smooth_item, window=3)

selected_smooth_path = TABLES_DIR / "06_selected_site_smoothed_temporal_forecast.csv"
selected_smooth_frame.to_csv(selected_smooth_path, index=False)

print("Выбранный участок:", selected_smooth_site)
print("Всего строк:", selected_smooth_item["n_total"])
print("Дат в окне прогноза:", selected_smooth_item["n_forecast_dates"])
print(f"MAE исходного прогноза = {selected_smooth_item['mae']:.3f}")
print(f"RMSE исходного прогноза = {selected_smooth_item['rmse']:.3f}")
print(f"R² исходного прогноза = {selected_smooth_item['r2']:.3f}")
print(f"MAE сглаженного графика = {selected_smooth_metrics['smooth_mae']:.3f}")
print(f"RMSE сглаженного графика = {selected_smooth_metrics['smooth_rmse']:.3f}")
print(f"R² сглаженного графика = {selected_smooth_metrics['smooth_r2']:.3f}")
print("Сохранено:", selected_smooth_path)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(
    selected_smooth_frame["date_end"],
    selected_smooth_frame["actual"],
    marker="o",
    linewidth=1,
    alpha=0.35,
    label="Факт, агрегированные значения",
)

ax.plot(
    selected_smooth_frame["date_end"],
    selected_smooth_frame["actual_smooth"],
    marker="o",
    linewidth=2.5,
    label="Факт, сглаженная тенденция",
)

ax.plot(
    selected_smooth_pred["date_end"],
    selected_smooth_pred["predicted_smooth"],
    marker="s",
    linewidth=2.5,
    linestyle="--",
    label="Прогноз нейросетевой модели",
)

ax.axvspan(
    selected_smooth_pred["date_end"].min(),
    selected_smooth_pred["date_end"].max(),
    alpha=0.12,
    label="Окно прогноза",
)

ax.axvline(
    selected_smooth_pred["date_end"].min(),
    linestyle=":",
    linewidth=2,
    alpha=0.8,
)

ax.set_title(
    f"Сглаженный временной ряд и прогноз нейросетевой модели\n"
    f"Участок: {selected_smooth_site}"
)
ax.set_xlabel("Дата окончания интервала")
ax.set_ylabel("Интенсивность изменения бровки, м/год")
ax.grid(alpha=0.3)
ax.legend(loc="best")

metrics_text = (
    f"MAE = {selected_smooth_metrics['smooth_mae']:.3f}\n"
    f"RMSE = {selected_smooth_metrics['smooth_rmse']:.3f}\n"
    f"R² = {selected_smooth_metrics['smooth_r2']:.3f}"
)

ax.text(
    0.78,
    0.95,
    metrics_text,
    transform=ax.transAxes,
    verticalalignment="top",
    bbox=dict(boxstyle="round", alpha=0.15),
)

output_path = FIGURES_DIR / "06_selected_site_smoothed_temporal_forecast.png"
fig.savefig(output_path, dpi=240, bbox_inches="tight", facecolor="white")

plt.show()

print("Сохранено:", output_path)


## 13. Ограничение сглаженного графика

Сглаженный график используется только как отчётная визуализация. Он показывает общую тенденцию и делает результат читаемым, но не является заменой исходных наблюдений.

Основные метрики качества модели нужно оценивать по несглаженным тестовым значениям. Сглаженные метрики можно использовать только как вспомогательную характеристику визуального совпадения тенденций.


## 14. Интерполированный временной ряд для отчётного графика

В этом блоке строится интерполированный ряд по выбранному участку.

Интерполяция применяется только для визуализации: она делает график более читаемым и показывает непрерывную траекторию между фактическими датами наблюдений. Она не используется как источник новых реальных наблюдений и не заменяет исходные данные.

Фактические точки остаются на графике отдельно, чтобы было видно, где находятся реальные наблюдения.


In [ ]:
INTERPOLATION_MANUAL_SITE_ID = None

if INTERPOLATION_MANUAL_SITE_ID is None:
    if "selected_smooth_site" in globals():
        interpolation_site = selected_smooth_site
    elif "selected_site_for_time_plot" in globals():
        interpolation_site = selected_site_for_time_plot
    elif len(temporal_candidates) > 0:
        interpolation_site = temporal_candidates[0]["site_id"]
    else:
        raise ValueError("Нет участка для интерполяции.")
else:
    interpolation_site = INTERPOLATION_MANUAL_SITE_ID

interpolation_item = build_temporal_forecast_for_site(interpolation_site)

if interpolation_item is None:
    raise ValueError(f"Не удалось построить временной прогноз для участка: {interpolation_site}")

interpolation_frame = interpolation_item["aggregated"].copy()
interpolation_frame["date_end"] = pd.to_datetime(interpolation_frame["date_end"], errors="coerce")
interpolation_frame = interpolation_frame.dropna(subset=["date_end"]).sort_values("date_end").reset_index(drop=True)

interpolation_frame


In [ ]:
def build_interpolated_series(frame):
    observed = frame[["date_end", "actual", "predicted"]].copy()
    observed = observed.dropna(subset=["date_end", "actual"])
    observed = observed.groupby("date_end", as_index=False).agg(
        actual=("actual", "median"),
        predicted=("predicted", "median"),
    )
    observed = observed.sort_values("date_end").reset_index(drop=True)

    date_min = observed["date_end"].min()
    date_max = observed["date_end"].max()

    annual_index = pd.date_range(
        start=pd.Timestamp(year=date_min.year, month=12, day=31),
        end=pd.Timestamp(year=date_max.year, month=12, day=31),
        freq="YE",
    )

    combined_index = pd.DatetimeIndex(
        sorted(set(annual_index).union(set(observed["date_end"])))
    )

    indexed = observed.set_index("date_end").reindex(combined_index)
    indexed.index.name = "date_end"

    indexed["actual_interpolated"] = indexed["actual"].interpolate(method="time")
    indexed["predicted_interpolated"] = indexed["predicted"].interpolate(method="time", limit_area="inside")

    forecast_start = observed.loc[observed["predicted"].notna(), "date_end"].min()
    forecast_end = observed.loc[observed["predicted"].notna(), "date_end"].max()

    if pd.notna(forecast_start) and pd.notna(forecast_end):
        mask = (indexed.index >= forecast_start) & (indexed.index <= forecast_end)
        indexed.loc[~mask, "predicted_interpolated"] = np.nan

    annual = indexed.loc[indexed.index.isin(annual_index)].copy()
    annual = annual.reset_index()

    return observed, annual


observed_points, interpolated_annual = build_interpolated_series(interpolation_frame)

observed_points_path = TABLES_DIR / "06_interpolation_observed_points.csv"
interpolated_annual_path = TABLES_DIR / "06_interpolation_annual_series.csv"

observed_points.to_csv(observed_points_path, index=False)
interpolated_annual.to_csv(interpolated_annual_path, index=False)

print("Участок:", interpolation_site)
print("Фактических дат:", len(observed_points))
print("Лет в интерполированном ряду:", len(interpolated_annual))
print("Сохранено:", observed_points_path)
print("Сохранено:", interpolated_annual_path)

interpolated_annual.head()


In [ ]:
forecast_annual = interpolated_annual.loc[
    interpolated_annual["predicted_interpolated"].notna()
].copy()

if len(forecast_annual) >= 2:
    interpolation_mae = mean_absolute_error(
        forecast_annual["actual_interpolated"],
        forecast_annual["predicted_interpolated"],
    )
    interpolation_rmse = np.sqrt(
        mean_squared_error(
            forecast_annual["actual_interpolated"],
            forecast_annual["predicted_interpolated"],
        )
    )

    if forecast_annual["actual_interpolated"].nunique() > 1:
        interpolation_r2 = r2_score(
            forecast_annual["actual_interpolated"],
            forecast_annual["predicted_interpolated"],
        )
    else:
        interpolation_r2 = np.nan
else:
    interpolation_mae = np.nan
    interpolation_rmse = np.nan
    interpolation_r2 = np.nan

print(f"MAE интерполированного графика = {interpolation_mae:.3f}")
print(f"RMSE интерполированного графика = {interpolation_rmse:.3f}")
print(f"R² интерполированного графика = {interpolation_r2:.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(
    interpolated_annual["date_end"],
    interpolated_annual["actual_interpolated"],
    linewidth=2.5,
    label="Факт, интерполированный ряд",
)

ax.scatter(
    observed_points["date_end"],
    observed_points["actual"],
    s=45,
    label="Фактические наблюдения",
)

if len(forecast_annual) > 0:
    ax.plot(
        forecast_annual["date_end"],
        forecast_annual["predicted_interpolated"],
        marker="s",
        linewidth=2.5,
        linestyle="--",
        label="Прогноз нейросетевой модели",
    )

    ax.axvspan(
        forecast_annual["date_end"].min(),
        forecast_annual["date_end"].max(),
        alpha=0.12,
        label="Окно прогноза",
    )

    ax.axvline(
        forecast_annual["date_end"].min(),
        linestyle=":",
        linewidth=2,
        alpha=0.8,
    )

ax.set_title(
    f"Интерполированный временной ряд и прогноз нейросетевой модели\n"
    f"Участок: {interpolation_site}"
)
ax.set_xlabel("Год")
ax.set_ylabel("Интенсивность изменения бровки, м/год")
ax.grid(alpha=0.3)
ax.legend(loc="best")

metrics_text = (
    f"MAE = {interpolation_mae:.3f}\n"
    f"RMSE = {interpolation_rmse:.3f}\n"
    f"R² = {interpolation_r2:.3f}"
)

ax.text(
    0.78,
    0.95,
    metrics_text,
    transform=ax.transAxes,
    verticalalignment="top",
    bbox=dict(boxstyle="round", alpha=0.15),
)

output_path = FIGURES_DIR / "06_selected_site_interpolated_temporal_forecast.png"
fig.savefig(output_path, dpi=240, bbox_inches="tight", facecolor="white")

plt.show()

print("Сохранено:", output_path)


## 15. Ограничение интерполяции

Интерполяция использована только для построения отчётного графика. Она создаёт промежуточные значения между датами наблюдений, но эти значения не являются новыми фактическими измерениями.

Поэтому интерполированный график можно использовать как визуализацию тенденции, но нельзя использовать как доказательство высокой точности модели на реальных ежегодных наблюдениях.

В отчёте рядом с таким графиком нужно указать, что непрерывный ряд получен интерполяцией по исходным датам наблюдений.
